# Impoort Libraries

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import math


# Set device and seed

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device:{device}")

# For reproducibility set seed
torch.manual_seed(43)

Using device:cpu


# Define the Multi head Attention class

In [3]:
# toy "stories"
text_data = """
Once upon a time there was a cat.
The cat loved to sleep all day.
Then it dreamed of being a tiger.
The tiger chased the stars in the sky.
One morning the cat woke up and purred.
The cat met a dog and they became friends.
Together they explored the garden and found a butterfly.
The butterfly led them to a hidden pond with fish.
They played near the pond until the sun set.
At night, the cat and dog slept under the stars.
The next day, they went on a little adventure through the forest.
The forest was full of birds and singing sounds.
A wise old owl told them stories about the trees.
They discovered a secret treehouse in the middle of the forest.
The treehouse had a small library of old books.
They spent hours reading tales of magic and dragons.
Suddenly, a gentle rain began, and they ran back home.
The cat drank milk, the dog chewed a bone, and they slept peacefully.
The next morning, the garden was full of colorful flowers.
A little mouse joined their morning playtime.
They built tiny bridges and houses for the mouse.
Finally, as the sun set, they rested again under the twinkling stars.
"""

# Convert to lowercase
text_data = text_data.lower()


# Tokenization

In [1]:
!pip3 install tiktoken

In [4]:

import tiktoken
import json

tokenizer = tiktoken.get_encoding("gpt2")
tokens = tokenizer.encode(text_data)
vocab_size = tokenizer.n_vocab


vocab = {i: tokenizer.decode([i]) for i in range(tokenizer.n_vocab)}

with open("gpt2_vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print("Vocab saved! Size:", len(vocab))


print("Total tokens:", len(tokens))
print("Vocab size:", vocab_size)


Vocab saved! Size: 50257
Total tokens: 269
Vocab size: 50257


# Create Input output pairs

In [8]:
# Convert tokens to tensor
data = torch.tensor(tokens, dtype=torch.long)

block_size = 32   # context window
batch_size = 8

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y


# Define the model

In [9]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.key = nn.Linear(n_embd, n_embd)
        self.query = nn.Linear(n_embd, n_embd)
        self.value = nn.Linear(n_embd, n_embd)
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        B, T, C = x.size()
        head_dim = C // self.n_head

        k = self.key(x).view(B, T, self.n_head, head_dim).transpose(1, 2)
        q = self.query(x).view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_head, head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)

        # causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device))
        attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(attn_scores, dim=-1)

        out = attn_weights @ v  # weighted sum
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


# Transformer Block

In [10]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


In [12]:
# Full Tiny GPT model

In [11]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=3, block_size=32):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_emb(idx)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx


In [14]:
# Training ........

In [12]:
model = TinyGPT(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(1000):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"Step {step}, loss {loss.item():.4f}")

        # --- SAVE CHECKPOINT ---
       # save_path = f"tinygpt_step_{step}.pth"
       # torch.save(model.state_dict(), save_path)
       # print(f"Saved model checkpoint → {save_path}")

# Save final model
torch.save(model.state_dict(), "tinygpt_model_final.pth")
print("Final model saved.")



Step 0, loss 10.9480
Step 50, loss 6.6131
Step 100, loss 3.5767
Step 150, loss 2.1394
Step 200, loss 1.4895
Step 250, loss 0.7226
Step 300, loss 0.3035
Step 350, loss 0.1843
Step 400, loss 0.1441
Step 450, loss 0.1320
Step 500, loss 0.0736
Step 550, loss 0.1089
Step 600, loss 0.0816
Step 650, loss 0.1136
Step 700, loss 0.0638
Step 750, loss 0.0476
Step 800, loss 0.0489
Step 850, loss 0.0683
Step 900, loss 0.0800
Step 950, loss 0.0426
Final model saved.


# Generate

In [21]:
# Start with first 10 tokens as context
context = torch.tensor([tokens[:20]], dtype=torch.long)

# Decode and print the context
print("Context or prompt (input to model):\n")
print(tokenizer.decode(context[0].tolist()))
print("\n" + "="*50 + "\n")

# Generate continuation
generated = model.generate(context, max_new_tokens=50)[0].tolist()

# Decode and print the generated text
print("Generated text:\n")
print(tokenizer.decode(generated))


Context or prompt (input to model):


once upon a time there was a cat.
the cat loved to sleep all day.



Generated text:


once upon a time there was a cat.
the cat loved to sleep all day.
then it dreamed of being a tiger.
the tiger chased the stars in the sky.
one morning the cat purred.
the cat met a dog and they became friends.
together they explored the garden and found a butterfly led them to
